# NIHONGA ? Phase 3: Identify Layers to Remove

**Phase registered:** `Phase 3 ? Identify Layers to Remove` (see [PLAN.md](PLAN.md#phase-3--identify-layers-to-remove)).

This phase will measure which transformer blocks appear redundant before any model is modified. We will begin conservatively, using the frozen Phase 2 evaluation conditions and locally saved artifacts so previously completed work is reused on reruns. Candidate importance signals include activation similarity across consecutive blocks, hidden-state cosine similarity, and the output change caused by bypassing one block. The resulting measurements will support selecting groups of layers for later pruning; this notebook will not prune or alter model weights.

This opening cell records the scope and safeguards for Phase 3.

## 1. Reuse the frozen Phase 2 model identity

Before inspecting any layers, we need to bind this analysis to the exact model used for the baseline. Phase 2 saved the model repository and immutable revision in `data/phase2/manifest.json` and repeated the evaluation settings in `data/phase2/protocol.json`. We will read those local files as the source of truth, then record their identities in the Phase 3 artifacts. This keeps the analysis reproducible and avoids a model download or any inference just to identify the baseline.

In [ ]:
# Read the frozen Phase 2 identity and persist Phase 3 provenance once
import hashlib
import json
from pathlib import Path

PHASE2_DIR = Path("data/phase2")
PHASE3_DIR = Path("data/phase3")
PHASE3_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = PHASE2_DIR / "manifest.json"
protocol_path = PHASE2_DIR / "protocol.json"
for required_path in (manifest_path, protocol_path):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required Phase 2 artifact is missing: {required_path}")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
model = manifest.get("model", {})
repo = model.get("repo") or protocol.get("MODEL_REPO")
revision = model.get("revision") or protocol.get("MODEL_REVISION")
if not repo or not revision:
    raise ValueError("Phase 2 artifacts do not identify both the model repository and immutable revision.")
if protocol.get("MODEL_REPO") not in (None, repo) or protocol.get("MODEL_REVISION") not in (None, revision):
    raise ValueError("Phase 2 manifest and protocol identify different model revisions.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

expected = {"model_repo": repo, "model_revision": revision,
            "phase2_manifest_sha256": sha256_file(manifest_path),
            "phase2_protocol_sha256": sha256_file(protocol_path)}
provenance_path = PHASE3_DIR / "provenance.json"
if provenance_path.exists():
    provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
    if any(provenance.get(key) != value for key, value in expected.items()):
        raise RuntimeError(f"Existing provenance does not match current Phase 2 inputs: {provenance_path}")
    print(f"Reusing verified provenance: {provenance_path}")
else:
    provenance = {"version": "phase3-provenance-v1", **expected}
    provenance_path.write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
    print(f"Saved provenance: {provenance_path}")

print(f"Model: {repo}@{revision}")
print(f"Baseline protocol: {protocol.get('PROTOCOL_VERSION', 'unspecified')}")

## 2. Inspect the saved transformer configuration

Layer importance measurements need the actual transformer depth and block layout for the pinned model. We will first look for a configuration file already present in the local Hugging Face cache, using the repository and revision recorded in Phase 3 provenance. Reading configuration metadata does not load model weights or execute inference. If that exact revision is not cached, this inspection should stop and report the missing local artifact rather than silently downloading a different or unpinned model. The discovered configuration and its checksum will be saved so later notebook runs can reuse the same architecture record.

In [ ]:
# Find the pinned model configuration in the local Hugging Face cache only
import hashlib
import json
import os
from pathlib import Path

PHASE3_DIR = Path("data/phase3")
provenance = json.loads((PHASE3_DIR / "provenance.json").read_text(encoding="utf-8"))
repo = provenance["model_repo"]
revision = provenance["model_revision"]
repo_cache_name = "models--" + repo.replace("/", "--")
hub_cache = Path(os.environ.get("HF_HUB_CACHE", Path.home() / ".cache" / "huggingface" / "hub"))
snapshot_dir = hub_cache / repo_cache_name / "snapshots" / revision

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

config_path = snapshot_dir / "transformer" / "config.json"
if not config_path.is_file():
    raise FileNotFoundError(
        f"Pinned transformer config is not available in the local cache ({config_path}). No download was attempted."
    )
config_bytes = config_path.read_bytes()
config = json.loads(config_bytes.decode("utf-8"))
config_sha256 = hashlib.sha256(config_bytes).hexdigest()

# Model configurations vary in naming. Preserve all metadata and report common depth keys.
depth_keys = ("num_layers", "num_hidden_layers", "depth", "transformer_layers", "layers")
depth_fields = {key: config[key] for key in depth_keys if key in config and isinstance(config[key], (int, float, list, dict))}
if "num_layers" not in depth_fields:
    raise ValueError(f"Transformer config does not expose a recognized layer count: {config_path}")
architecture_record = {
    "version": "phase3-architecture-config-v1",
    "model_repo": repo,
    "model_revision": revision,
    "config_relative_path": config_path.relative_to(snapshot_dir).as_posix(),
    "config_sha256": config_sha256,
    "depth_fields": depth_fields,
    "config": config,
}

config_snapshot = PHASE3_DIR / "model_config.json"
architecture_path = PHASE3_DIR / "architecture.json"
if architecture_path.exists() or config_snapshot.exists():
    if not (architecture_path.is_file() and config_snapshot.is_file()):
        raise RuntimeError("Phase 3 architecture artifacts are incomplete; inspect them before rerunning.")
    saved_architecture = json.loads(architecture_path.read_text(encoding="utf-8"))
    saved_config = json.loads(config_snapshot.read_text(encoding="utf-8"))
    if saved_architecture == architecture_record and saved_config == config:
        print(f"Reusing verified architecture artifacts in {PHASE3_DIR}")
    elif (saved_architecture.get("model_repo") == repo
          and saved_architecture.get("model_revision") == revision
          and saved_architecture.get("config_relative_path") != architecture_record["config_relative_path"]):
        # Replace the earlier component config selection with this verified transformer config.
        config_snapshot.write_text(json.dumps(config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        architecture_path.write_text(json.dumps(architecture_record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        print(f"Corrected architecture artifacts to use {architecture_record['config_relative_path']}")
    else:
        raise RuntimeError("Saved Phase 3 architecture record differs from the pinned local transformer config.")
else:
    config_snapshot.write_text(json.dumps(config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    architecture_path.write_text(json.dumps(architecture_record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Saved pinned transformer config snapshot and architecture summary in {PHASE3_DIR}")

print(f"Config: {architecture_record['config_relative_path']} (sha256 {config_sha256})")
print(f"Common depth fields: {depth_fields or 'none found; inspect saved config structure'}")

## 3. Freeze a reproducible layer-analysis probe set

Layer bypass and hidden-state comparisons must use identical prompts, seeds, resolutions, and sampling settings. We will derive a small diagnostic subset from the already frozen Phase 2 internal evaluation set, stratifying deterministically across its capability dimensions and difficulty levels. Each selected row will retain its original generation conditions and metadata. We will serialize the selected rows and their source-file hash under `data/phase3/`; reruns will reuse the saved subset and verify that it still matches the Phase 2 source. This selection step makes no model calls and does not alter the baseline evaluation set.

In [ ]:
# Build and persist a deterministic stratified probe set from Phase 2
import hashlib
import json
from collections import defaultdict
from pathlib import Path

PHASE2_EVAL_PATH = Path("data/phase2/internal_eval_set.jsonl")
PHASE3_DIR = Path("data/phase3")
PROBE_PATH = PHASE3_DIR / "layer_probe_set.jsonl"
PROBE_MANIFEST_PATH = PHASE3_DIR / "layer_probe_manifest.json"
PROBE_LIMIT = 64
PROBE_SALT = "nihonga-phase3-layer-probe-v1"
if not PHASE2_EVAL_PATH.is_file():
    raise FileNotFoundError(f"Frozen Phase 2 internal evaluation set not found: {PHASE2_EVAL_PATH}")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_sha256 = sha256_file(PHASE2_EVAL_PATH)
rows = [json.loads(line) for line in PHASE2_EVAL_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
if not rows:
    raise ValueError("Phase 2 internal evaluation set is empty.")
required_fields = {"eval_id", "prompt", "seed", "width", "height", "num_inference_steps", "true_cfg_scale"}
missing = sorted(required_fields.difference(rows[0]))
if missing:
    raise ValueError(f"Phase 2 evaluation rows are missing required conditions: {missing}")

groups = defaultdict(list)
for row in rows:
    key = (str(row.get("dimension", "unknown")), str(row.get("difficulty", "unknown")))
    groups[key].append(row)
for key in groups:
    groups[key].sort(key=lambda row: hashlib.sha256(f"{PROBE_SALT}:{row['eval_id']}".encode()).hexdigest())

selected = []
ordered_strata = sorted(groups)
while len(selected) < min(PROBE_LIMIT, len(rows)):
    added_this_round = False
    for key in ordered_strata:
        already_selected = sum(1 for row in selected if (str(row.get("dimension", "unknown")), str(row.get("difficulty", "unknown"))) == key)
        if already_selected < len(groups[key]):
            selected.append(groups[key][already_selected])
            added_this_round = True
            if len(selected) >= min(PROBE_LIMIT, len(rows)):
                break
    if not added_this_round:
        break

probe_manifest = {
    "version": "phase3-layer-probe-v1",
    "source_path": PHASE2_EVAL_PATH.as_posix(),
    "source_sha256": source_sha256,
    "selection_salt": PROBE_SALT,
    "selection_method": "sha256-ranked round-robin across (dimension, difficulty) strata",
    "requested_max_rows": PROBE_LIMIT,
    "selected_rows": len(selected),
    "strata_counts": {f"{dimension}|difficulty={difficulty}": sum(1 for row in selected if str(row.get("dimension", "unknown")) == dimension and str(row.get("difficulty", "unknown")) == difficulty) for dimension, difficulty in ordered_strata},
    "eval_ids": [row["eval_id"] for row in selected],
}

PHASE3_DIR.mkdir(parents=True, exist_ok=True)
if PROBE_PATH.exists() or PROBE_MANIFEST_PATH.exists():
    if not (PROBE_PATH.is_file() and PROBE_MANIFEST_PATH.is_file()):
        raise RuntimeError("Probe artifacts are incomplete; inspect them before rerunning.")
    saved_manifest = json.loads(PROBE_MANIFEST_PATH.read_text(encoding="utf-8"))
    saved_rows = [json.loads(line) for line in PROBE_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    if saved_manifest != probe_manifest or saved_rows != selected:
        raise RuntimeError("Saved probe set differs from the current frozen Phase 2 source or deterministic selection.")
    print(f"Reusing verified probe set: {len(saved_rows)} rows from {PROBE_PATH}")
else:
    PROBE_PATH.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in selected), encoding="utf-8")
    PROBE_MANIFEST_PATH.write_text(json.dumps(probe_manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Saved deterministic probe set: {len(selected)} rows to {PROBE_PATH}")

print(f"Source SHA-256: {source_sha256}")
print(f"Strata represented: {len(ordered_strata)}; probe cap: {PROBE_LIMIT}")

## 4. Check the local analysis runtime and model weights

Before collecting activation or bypass measurements, confirm that this notebook runtime has the needed inference packages and that the pinned transformer weights exist in the same local model snapshot as the verified config. This preflight records Python and package versions, accelerator availability, and the relevant weight-file inventory. It does not load model weights, run inference, or download files. Saving the report makes later measurements traceable to the environment that produced them.